# CEO Dashboard Crisis & AI Decision Scenario (Module 14)

## Objective
Demonstrate a real-world enterprise situation showing how the platform detects a drop in revenue, a spike in customer churn, and inventory shortages, explains the root causes using SHAP, and outputs actionable recommendations with confidence and estimated ROI.

### Narrative Context

It is Monday morning. The CEO opens the platform's Executive Dashboard and notices:
1. **Revenue Drop**: Next-month revenue forecast is dropping by 8%.
2. **Churn Spike**: Customer churn flags have increased, specifically in high-value Mid-Market accounts.
3. **Inventory Alert**: Stockout risk has escalated on critical category products.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
backend_root = project_root / 'backend'
sys.path.insert(0, str(backend_root))

from app.ml.revenue_forecasting import RevenueForecastingPaths, RevenueForecastingService
from app.ml.churn_prediction import ChurnPredictionPaths, ChurnPredictionService
from app.ml.demand_forecasting import DemandForecastingPaths, DemandForecastingService
from app.ml.business_risk import BusinessRiskPaths, BusinessRiskService
from app.ml.explainability import ExplainabilityService
from app.ml.recommendation_engine import BusinessRecommendationService


## Phase 1: Anomaly and Risk Detection

Let's load the latest finance monthly metrics, check the upcoming forecast, and count the customer churn/risk states.

In [ ]:
exports_dir = project_root / 'exports'
reports_dir = project_root / 'reports'
features_dir = project_root / 'features'

# 1. Load revenue forecast
rev_forecast = pd.read_csv(exports_dir / 'revenue_forecast.csv')
last_month = rev_forecast.iloc[-1]
print(f"Current Month Revenue: ${last_month['actual_current_month_revenue']:,.2f}")
print(f"Forecast Next Month:   ${last_month['predicted_next_month_revenue']:,.2f}")
change = ((last_month['predicted_next_month_revenue'] - last_month['actual_current_month_revenue']) / last_month['actual_current_month_revenue']) * 100
print(f"Revenue Trend: {change:.1f}%")

# 2. Load churn counts
churn_preds = pd.read_csv(exports_dir / 'customer_churn_predictions.csv')
total_churn = churn_preds['predicted_churn'].sum()
print(f"Number of predicted churn customers: {total_churn} out of {len(churn_preds)} ({total_churn/len(churn_preds)*100:.1f}%)")

## Phase 2: Explainable AI Root Cause Analysis

The CEO demands to know: *why is churn spiking, and what features are causing the revenue drop?*
We load the SHAP service and look at the top global impact metrics for churn.

In [ ]:
trained_models_dir = project_root / 'trained_models'
explain_service = ExplainabilityService(trained_models_dir=trained_models_dir, exports_dir=exports_dir)

customer_features = pd.read_csv(features_dir / 'customer_features.csv')
global_exps = explain_service.explain_global('churn_prediction_model', customer_features.head(100))

print("Top 5 Churn Drivers (SHAP Importance):")
pd.DataFrame(global_exps['global_importance']).head(5)

## Phase 3: Prescriptive Action Recommendations

Using the detected risks and SHAP explanations, the Recommendation Engine creates business actions. Each action provides a confidence score, priority, ROI, and expected time to benefit.

In [ ]:
rec_service = BusinessRecommendationService(exports_dir=exports_dir, reports_dir=reports_dir)
recommendations = rec_service.generate_recommendations()
pd.DataFrame([dict(r.__dict__) for r in recommendations])

## Summary

This scenario showcases the power of the platform: it doesn't just show charts; it alerts key leaders to revenue drops, points out the churn spike in high-value accounts, explains *why* it is happening via SHAP, and outlines immediate steps to restore margins and retain customers with positive estimated ROI.